In [14]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
import os 
import re
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             accuracy_score, precision_score, recall_score, f1_score)
from sklearn.model_selection import ParameterGrid
from sklearn.base import clone
from sklearn.pipeline import Pipeline

df = pd.read_csv("../data/training_data/mixed_demographics.csv")
#df["MI"].value_counts() #healthy = 452, pMI = 448
#df.isna().sum() #no missing values

### Identify impossible anatomies

In [15]:
bad_folder = "../data/training_data/impossible_samples"
bad_files = [f for f in os.listdir(bad_folder) if f.endswith(".npy")]
bad_indices = [int(re.match(r"\d+", f).group()) for f in bad_files]

print("Number of bad subjects:", len(bad_indices))
print(sorted(bad_indices))

Number of bad subjects: 38
[57, 75, 113, 138, 139, 172, 174, 289, 292, 308, 326, 335, 378, 385, 400, 424, 426, 439, 445, 503, 508, 573, 591, 601, 658, 666, 673, 693, 699, 722, 754, 776, 793, 814, 822, 841, 846, 875]


### Remove impossible anatomies

In [16]:
folder = "../data/training_data/clean_samples"
files = [f for f in os.listdir(folder) if f.endswith(".npy")]
files = sorted(files, key=lambda f: int(re.match(r"\d+", f).group()))

df_clean = df[~df["index"].isin(bad_indices)].copy()

file_ids = [int(re.match(r"\d+", f).group()) for f in files]

df_clean = df_clean.set_index("index").loc[file_ids].reset_index()

assert df_clean["index"].tolist() == file_ids

print("Original dataset:", df.shape)
print("Clean dataset:", df_clean.shape)

print("Bad subjects remaining:", df_clean["index"].isin(bad_indices).sum())

Original dataset: (900, 9)
Clean dataset: (862, 9)
Bad subjects remaining: 0


### Split X, y into train and test

In [17]:
features = ["age", "BMI", "height", "weight", "diastolic_BP", "systolic_BP", "sex"]

X = df_clean[features].copy()
y = df_clean["MI"].map({"healthy": 0, "pMI": 1})

X_dev, X_test, files_dev, files_test, y_dev, y_test = train_test_split(
    X, files, y, test_size=150, random_state=9888, stratify=y)

print("Development shape:", X_dev.shape, "Test shape:", X_test.shape)
print("Development labels:\n", y_dev.value_counts())
print("Test labels:\n", y_test.value_counts())

Development shape: (712, 7) Test shape: (150, 7)
Development labels:
 MI
0    360
1    352
Name: count, dtype: int64
Test labels:
 MI
0    76
1    74
Name: count, dtype: int64


### Load geometry files

In [18]:
def load_geometry(filenames):
    coords, regions, motion_features = [], [], []
    for f in filenames:
        sample = np.load(os.path.join(folder, f), mmap_mode="r")
        xyz = np.asarray(sample[:, :, :3], dtype=np.float64)

        distances = np.linalg.norm(np.diff(xyz, axis=0), axis=2)
        stats = [xyz.mean(), xyz.std(), distances.mean(), distances.std(), distances.sum(axis=0).mean()]
        motion_features.append(stats)
        coords.append(xyz.ravel())
        regions.append(np.asarray(sample[:, :, 3]).ravel())
    return np.stack(coords), np.stack(regions), np.asarray(motion_features)

motion_features = ["mean_position", "std_position", "mean_displacement",
                   "std_displacement", "mean_total_distance"]
coords_dev, regions_dev, motion_dev = load_geometry(files_dev)
coords_test, regions_test, motion_test = load_geometry(files_test)
print("Development coordinates:", coords_dev.shape)
print("Movement features:", motion_features)
print("Development movement shape:", motion_dev.shape)

Development coordinates: (712, 540000)
Movement features: ['mean_position', 'std_position', 'mean_displacement', 'std_displacement', 'mean_total_distance']
Development movement shape: (712, 5)


### Five-fold Cross Validation

In [19]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=9888)
results = []

def prepare_features(train_coords, valid_coords, train_regions, valid_regions, 
                     train_demographics, valid_demographics, train_motion, valid_motion):
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    encoder.fit(train_regions.reshape(-1, 1))
    train_encoded = encoder.transform(train_regions.reshape(-1, 1)).reshape(len(train_regions), -1)
    valid_encoded = encoder.transform(valid_regions.reshape(-1, 1)).reshape(len(valid_regions), -1)

    train_geometry = np.hstack([train_coords, train_encoded])
    valid_geometry = np.hstack([valid_coords, valid_encoded])

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_geometry)
    valid_scaled = scaler.transform(valid_geometry)

    pca = PCA(n_components=7, random_state=9888)
    train_pca = pca.fit_transform(train_scaled)
    valid_pca = pca.transform(valid_scaled)

    train_features = np.hstack([train_demographics, train_pca, train_motion])
    valid_features = np.hstack([valid_demographics, valid_pca, valid_motion])
    return train_features, valid_features, pca

models = {
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=9888, n_jobs=-1),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=9888),
    "Gradient Boosted Trees": GradientBoostingClassifier(random_state=9888),
}

all_results = []
for fold, (train_idx, val_idx) in enumerate(cv.split(coords_dev, y_dev), start=1):
    train_features, val_features, _ = prepare_features(
        coords_dev[train_idx], coords_dev[val_idx],
        regions_dev[train_idx], regions_dev[val_idx],
        X_dev.iloc[train_idx][features].to_numpy(dtype=float),
        X_dev.iloc[val_idx][features].to_numpy(dtype=float),
        motion_dev[train_idx], motion_dev[val_idx]
    )
    final_scaler = StandardScaler()
    train_lr = final_scaler.fit_transform(train_features)
    val_lr = final_scaler.transform(val_features)

    for name, model in models.items():
        fit_x, eval_x = (train_lr, val_lr) if name == "Logistic Regression" else (train_features, val_features)
        model.fit(fit_x, y_dev.iloc[train_idx])
        pred = model.predict(eval_x)
        prob = model.predict_proba(eval_x)[:, 1]
        all_results.append({
            "model": name, "fold": fold,
            "accuracy": accuracy_score(y_dev.iloc[val_idx], pred),
            "precision": precision_score(y_dev.iloc[val_idx], pred, zero_division=0),
            "recall": recall_score(y_dev.iloc[val_idx], pred, zero_division=0),
            "f1": f1_score(y_dev.iloc[val_idx], pred, zero_division=0),
            "roc_auc": roc_auc_score(y_dev.iloc[val_idx], prob),
        })
    print(f"Fold {fold} complete")

comparison = pd.DataFrame(all_results)
print(comparison.groupby("model")[["accuracy", "precision", "recall", "f1", "roc_auc"]]
      .agg(["mean", "std"]).round(3))
    

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete
                       accuracy        precision        recall            f1  \
                           mean    std      mean    std   mean    std   mean   
model                                                                          
Gradient Boosted Trees    0.823  0.025     0.832  0.035  0.807  0.034  0.819   
Logistic Regression       0.768  0.025     0.768  0.035  0.764  0.030  0.765   
Random Forest             0.824  0.033     0.826  0.042  0.818  0.023  0.822   

                              roc_auc         
                          std    mean    std  
model                                         
Gradient Boosted Trees  0.025   0.905  0.015  
Logistic Regression     0.023   0.856  0.014  
Random Forest           0.031   0.906  0.016  


### Hyper-parameter Tuning

In [20]:
fold_data = []
for train_idx, val_idx in cv.split(coords_dev, y_dev):
    train_features, val_features, _ = prepare_features(
        coords_dev[train_idx], coords_dev[val_idx],
        regions_dev[train_idx], regions_dev[val_idx],
        X_dev.iloc[train_idx][features].to_numpy(dtype=float),
        X_dev.iloc[val_idx][features].to_numpy(dtype=float),
        motion_dev[train_idx], motion_dev[val_idx]
    )
    fold_data.append((train_features, val_features,
                      y_dev.iloc[train_idx], y_dev.iloc[val_idx]))

In [21]:
search = {
    "Logistic Regression": (
        Pipeline([("scaler", StandardScaler()),
                  ("clf", LogisticRegression(max_iter=2000, random_state=9888))]),
        {"clf__C": [0.1, 1, 10, 50, 100]}),
    "Random Forest": (
        RandomForestClassifier(random_state=9888, n_jobs=-1),
        {"n_estimators": [200, 300, 500], "max_depth": [None, 5, 10],
         "min_samples_leaf": [1, 3, 5], "max_features": ["sqrt", 0.5]}),
    "Gradient Boosted Trees": (
        GradientBoostingClassifier(random_state=9888),
        {"n_estimators": [100, 200, 300], "learning_rate": [0.03, 0.1, 0.3],
         "max_depth": [2, 3, 4, 5], "subsample": [0.6, 0.8, 1.0]}),
}

rows = []
for name, (est, grid) in search.items():
    for params in ParameterGrid(grid):
        aucs = []
        f1s = []
        for tr, va, ytr, yva in fold_data:
            m = clone(est).set_params(**params).fit(tr, ytr)
            aucs.append(roc_auc_score(yva, m.predict_proba(va)[:, 1]))
            f1s.append(f1_score(yva, m.predict(va), zero_division=0))
        rows.append({"model": name, "params": params,
                     "mean_auc": np.mean(aucs), "std_auc": np.std(aucs),
                     "mean_f1": np.mean(f1s), "std_f1": np.std(f1s)})
tuning = pd.DataFrame(rows)

best_params = {}
for name in search:
    top = tuning[tuning["model"] == name].sort_values("mean_auc", ascending=False).iloc[0]
    best_params[name] = top["params"]
    print(f"{name}: AUROC {top['mean_auc']:.3f} ± {top['std_auc']:.3f}   F1 {top['mean_f1']:.3f} ± {top['std_f1']:.3f}  {top['params']}")

Logistic Regression: AUROC 0.860 ± 0.013   F1 0.762 ± 0.021  {'clf__C': 100}
Random Forest: AUROC 0.907 ± 0.012   F1 0.830 ± 0.020  {'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 500}
Gradient Boosted Trees: AUROC 0.923 ± 0.011   F1 0.838 ± 0.014  {'learning_rate': 0.3, 'max_depth': 5, 'n_estimators': 300, 'subsample': 0.8}


### Test evaluation

In [22]:
# Fit preprocessing on all development subjects, then evaluate each model once on test subjects.
train_features, test_features, final_pca = prepare_features(
    coords_dev, coords_test, regions_dev, regions_test,
    X_dev[features].to_numpy(dtype=float),
    X_test[features].to_numpy(dtype=float),
    motion_dev, motion_test
)

# Scale final feature matrix for logistic regression (fit on development subjects only).
final_scaler = StandardScaler()
train_lr = final_scaler.fit_transform(train_features)
test_lr = final_scaler.transform(test_features)

for name, model in models.items():
    fit_x, eval_x = (train_lr, test_lr) if name == "Logistic Regression" else (train_features, test_features)
    model.fit(fit_x, y_dev)
    y_pred = model.predict(eval_x)
    y_prob = model.predict_proba(eval_x)[:, 1]
    print(f"\n===== {name} =====")
    print(classification_report(y_test, y_pred, target_names=["healthy", "pMI"], zero_division=0))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
    print("Test ROC AUC:", round(roc_auc_score(y_test, y_prob), 3))

print("PCA variance retained:", final_pca.explained_variance_ratio_.sum())
feature_names = features + [f"PC{i}" for i in range(1, 7 + 1)] + motion_features
print("Model features:", feature_names)


===== Random Forest =====
              precision    recall  f1-score   support

     healthy       0.84      0.89      0.87        76
         pMI       0.88      0.82      0.85        74

    accuracy                           0.86       150
   macro avg       0.86      0.86      0.86       150
weighted avg       0.86      0.86      0.86       150

Confusion matrix:
 [[68  8]
 [13 61]]
Test ROC AUC: 0.892

===== Logistic Regression =====
              precision    recall  f1-score   support

     healthy       0.77      0.79      0.78        76
         pMI       0.78      0.76      0.77        74

    accuracy                           0.77       150
   macro avg       0.77      0.77      0.77       150
weighted avg       0.77      0.77      0.77       150

Confusion matrix:
 [[60 16]
 [18 56]]
Test ROC AUC: 0.844

===== Gradient Boosted Trees =====
              precision    recall  f1-score   support

     healthy       0.81      0.89      0.85        76
         pMI       0.88  